# 3 · `data_forecast.ipynb`

Generate valid samples for the **forecast** task: past 15 min of images + PV (16 one-minute steps)
→ PV 15 minutes ahead. Then split into train / val / test and save.

**Needs in Drive:** `SKIPPD/images_pv_processed.hdf5` + `times_trainval.npy` + `times_test.npy`
(from notebook 2, or the official benchmark files).

> **Reconstruction note.** Clean, Colab-runnable reimplementation following the purpose the SKIPP'D
> README describes — not a byte-for-byte copy of the authors' notebook. **Image-only pipeline: no
> video processing.** All data lives in `/content/drive/MyDrive/SKIPPD/`.


## Mount Google Drive

Everything reads from and writes to `/content/drive/MyDrive/SKIPPD/`.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/SKIPPD"
os.makedirs(BASE, exist_ok=True)
print("Working folder:", BASE)
print("Contents:", os.listdir(BASE))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working folder: /content/drive/MyDrive/SKIPPD
Contents: ['2019_pv_raw.csv', 'pv_processed_10s.csv', 'images_raw', 'images_pv_processed.hdf5', 'times_test.npy', 'times_trainval.npy', 'model_output', 'forecast_samples.npz']


## Imports & config

In [5]:
import h5py, numpy as np
HDF5     = os.path.join(BASE, "images_pv_processed.hdf5")
TIMES_TV = os.path.join(BASE, "times_trainval.npy")
TIMES_TE = os.path.join(BASE, "times_test.npy")

SEQ_LEN = 16    # past frames t-15..t (1-min spacing)
HORIZON = 15    # predict PV 15 minutes ahead

In [6]:
with h5py.File(HDF5, "r") as f:
    img_tv, pv_tv = f["trainval/images_log"][:], f["trainval/pv_log"][:]
    img_te, pv_te = f["test/images_log"][:],     f["test/pv_log"][:]
t_tv = np.load(TIMES_TV, allow_pickle=True).astype("datetime64[m]")
t_te = np.load(TIMES_TE, allow_pickle=True).astype("datetime64[m]")
print("trainval:", img_tv.shape, "| test:", img_te.shape)

trainval: (2416, 64, 64, 3) | test: (427, 64, 64, 3)


## Build valid sliding-window samples

Each sample needs 16 consecutive 1-minute input steps and a target exactly 15 minutes after the last
input step; anything with a time gap is skipped.

In [7]:
def build_samples(imgs, pv, times, seq_len, horizon):
    times = times.astype("datetime64[m]")
    Xi, Xp, y = [], [], []
    for i in range(seq_len - 1, len(imgs) - horizon):
        s = i - seq_len + 1
        win = times[s:i + 1]
        if not np.all(np.diff(win).astype("timedelta64[m]").astype(int) == 1):
            continue
        if (times[i + horizon] - times[i]).astype("timedelta64[m]").astype(int) != horizon:
            continue
        Xi.append(imgs[s:i + 1]); Xp.append(pv[s:i + 1]); y.append(pv[i + horizon])
    if not Xi:
        return (np.empty((0, seq_len, 64, 64, 3), np.uint8),
                np.empty((0, seq_len), np.float32), np.empty((0,), np.float32))
    return np.stack(Xi), np.stack(Xp).astype(np.float32), np.array(y, np.float32)

Xi_tv, Xp_tv, y_tv = build_samples(img_tv, pv_tv, t_tv, SEQ_LEN, HORIZON)
Xi_te, Xp_te, y_te = build_samples(img_te, pv_te, t_te, SEQ_LEN, HORIZON)
print("trainval samples:", len(y_tv), "| test samples:", len(y_te))

trainval samples: 1949 | test samples: 367


## Split train / val and save to Drive

In [8]:
rng  = np.random.default_rng(0)
perm = rng.permutation(len(y_tv))
n_val = int(0.1 * len(y_tv))
val_idx, tr_idx = perm[:n_val], perm[n_val:]

OUT_NPZ = os.path.join(BASE, "forecast_samples.npz")
np.savez_compressed(OUT_NPZ,
    Xi_train=Xi_tv[tr_idx], Xp_train=Xp_tv[tr_idx], y_train=y_tv[tr_idx],
    Xi_val=Xi_tv[val_idx],  Xp_val=Xp_tv[val_idx],  y_val=y_tv[val_idx],
    Xi_test=Xi_te,          Xp_test=Xp_te,          y_test=y_te)
print("Saved ->", OUT_NPZ)
print("train:", len(tr_idx), "| val:", len(val_idx), "| test:", len(y_te))

Saved -> /content/drive/MyDrive/SKIPPD/forecast_samples.npz
train: 1755 | val: 194 | test: 367


In [9]:
# ============================================================================
#  Forecast — evaluation: regression metrics + (binned) classification
#  Baseline is PERSISTENCE (assume PV stays constant for 15 min).
# ============================================================================
import os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_recall_fscore_support)

BASE         = "/content/drive/MyDrive/SKIPPD"
FORECAST_DIR = os.path.join(BASE, "model_output", "SUNSET_forecast_run")
FORECAST_NPZ = os.path.join(BASE, "forecast_samples.npz")

# --- load forecast test data ---
d  = np.load(FORECAST_NPZ)
Xi = np.transpose(d["Xi_test"], (0, 2, 3, 1, 4))
Xi = Xi.reshape(Xi.shape[0], Xi.shape[1], Xi.shape[2], -1)   # (N,64,64,48)
Xp = d["Xp_test"].astype("float32")                          # (N,16) PV history
y_test = d["y_test"].astype("float32")

# --- load ensemble & predict (two inputs) ---
paths = sorted(glob.glob(os.path.join(FORECAST_DIR, "best_model_rep_*.keras")))
assert paths, "No forecast models found — train the forecast notebook first."
preds = [np.squeeze(keras.models.load_model(p).predict([Xi, Xp], batch_size=64, verbose=0))
         for p in paths]
y_pred  = np.clip(np.mean(preds, axis=0), 0, None)
persist = np.clip(Xp[:, -1], 0, None)                        # persistence baseline

# ---------- PART 1: REGRESSION METRICS ----------
err  = y_pred - y_test
mae  = np.mean(np.abs(err)); rmse = np.sqrt(np.mean(err**2)); mbe = np.mean(err)
r2   = 1 - np.sum(err**2)/np.sum((y_test - y_test.mean())**2)
corr = np.corrcoef(y_test, y_pred)[0, 1]
rmse_persist = np.sqrt(np.mean((persist - y_test)**2))
skill = 100*(1 - rmse/rmse_persist)                          # skill vs PERSISTENCE

print("="*50); print("  FORECAST REGRESSION METRICS"); print("="*50)
print(f"  MAE   : {mae:7.3f} kW")
print(f"  RMSE  : {rmse:7.3f} kW")
print(f"  MBE   : {mbe:7.3f} kW")
print(f"  R^2   : {r2:7.3f}")
print(f"  Corr  : {corr:7.3f}")
print(f"  Persistence RMSE : {rmse_persist:.3f} kW")
print(f"  Forecast SKILL   : {skill:6.2f} %  (>0 = beats persistence)")

# ---------- PART 2: CLASSIFICATION METRICS (bin into Low/Med/High) ----------
edges  = np.quantile(y_test, [0, 1/3, 2/3, 1.0])
labels = ["Low", "Medium", "High"]
to_class = lambda v: np.clip(np.digitize(v, edges[1:-1]), 0, 2)
t_cls, p_cls = to_class(y_test), to_class(y_pred)

acc = accuracy_score(t_cls, p_cls)
prec, rec, f1, sup = precision_recall_fscore_support(t_cls, p_cls, labels=[0,1,2], zero_division=0)
print("\n" + "="*50); print(f"  CLASSIFICATION METRICS (edges kW: {np.round(edges,1)})"); print("="*50)
print(f"  Accuracy: {acc:.3f}\n")
tbl = pd.DataFrame({"precision":prec,"recall":rec,"f1_score":f1,"support":sup}, index=labels).round(3)
print(tbl.to_string())
print("\n", classification_report(t_cls, p_cls, target_names=labels, zero_division=0))

# ---------- PART 3: CONFUSION MATRIX ----------
cm = confusion_matrix(t_cls, p_cls, labels=[0,1,2])
fig, ax = plt.subplots(figsize=(5,4)); im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3)); ax.set_xticklabels(labels)
ax.set_yticks(range(3)); ax.set_yticklabels(labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title(f"Forecast confusion matrix ({acc:.0%})")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i,j], ha="center", va="center",
                color="white" if cm[i,j] > cm.max()/2 else "black")
plt.colorbar(im); plt.tight_layout(); plt.show()

AssertionError: No forecast models found — train the forecast notebook first.

In [10]:
import os, glob
BASE = "/content/drive/MyDrive/SKIPPD"
print(glob.glob(os.path.join(BASE, "model_output", "SUNSET_forecast_run", "best_model_rep_*.keras")))

[]
